# 01 — Data Cleaning
Load the Loan Default dataset, inspect features, separate target, and save clean CSV.

**Dataset:** nikhil1e9/loan-default (Kaggle)  
**Rows:** 255,347 | **Columns:** 18

In [ ]:
import pandas as pd
import os
from src.data_loader import load_data
from src.config import (
    DATA_RAW_PATH, DATA_PROCESSED_PATH, TARGET, ID_COLUMN,
    NUMERICAL_FEATURES, CATEGORICAL_FEATURES
)

pd.set_option("display.max_columns", 20)

## 1. Load Full Dataset (No Sampling Needed)
The dataset is only 24.8 MB — small enough to load entirely.

In [ ]:
df = load_data(DATA_RAW_PATH)
print(f"
Shape: {df.shape}")
print(f"
Column types:
{df.dtypes}")
print(f"
First 3 rows:")
df.head(3)

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"
Total missing: {df.isnull().sum().sum()}")

## 2. Problem Type Identification
This is a **binary classification** problem.
- Target: `Default` — 0 (No Default) or 1 (Default)
- All 16 features are pre-loan-decision data (no leakage risk)

In [ ]:
print("Class distribution:")
print(df[TARGET].value_counts())
print(f"
Imbalance ratio: {df[TARGET].value_counts()[0] / df[TARGET].value_counts()[1]:.2f}:1")
print(f"Default rate: {df[TARGET].mean()*100:.1f}%")

## 3. Drop LoanID (Identifier, Not a Feature)

In [ ]:
print(f"Dropping "{ID_COLUMN}" (unique identifier, not a feature)")
print(f"Sample IDs: {df[ID_COLUMN].head(3).tolist()}")
print(f"Unique IDs: {df[ID_COLUMN].nunique():,} (should equal row count)")

df = df.drop(columns=[ID_COLUMN])
print(f"
Shape after dropping ID: {df.shape}")

## 4. Verify Feature Lists Match Config

In [ ]:
all_features = NUMERICAL_FEATURES + CATEGORICAL_FEATURES + [TARGET]
print(f"Expected columns ({len(all_features)}): {all_features}")
print(f"Actual columns ({len(df.columns)}): {df.columns.tolist()}")

missing = set(all_features) - set(df.columns)
extra = set(df.columns) - set(all_features)
print(f"
Missing from data: {missing or "None"}")
print(f"Extra in data: {extra or "None"}")

# Select only our features
df = df[all_features]
print(f"
Final shape: {df.shape}")

## 5. Descriptive Statistics

In [ ]:
print("Numerical features — summary statistics:")
df[NUMERICAL_FEATURES].describe().round(2)

In [ ]:
print("Categorical features — unique values:")
for col in CATEGORICAL_FEATURES:
    print(f"  {col}: {df[col].nunique()} categories → {df[col].unique().tolist()}")

## 6. Save Cleaned Data

In [ ]:
os.makedirs(os.path.dirname(DATA_PROCESSED_PATH), exist_ok=True)

df.to_csv(DATA_PROCESSED_PATH, index=False)
print(f"Saved to {DATA_PROCESSED_PATH}")
print(f"File size: {os.path.getsize(DATA_PROCESSED_PATH) / 1024 / 1024:.1f} MB")
print(f"Shape: {df.shape}")

## 7. Data Leakage — Concept Explanation

This dataset has **no leakage columns** — all features are pre-decision data.
But in many real-world credit datasets (e.g., Lending Club), columns like
`recoveries` or `last_payment_amount` are present. These values only exist
AFTER the loan outcome is determined:

- `recoveries > 0` → the borrower already defaulted (that is how recovery happens)
- Using such a feature gives ~99% accuracy but is **fraudulent** —
  it is using the answer to predict the answer.

**Rule:** If a feature can only be computed AFTER the event you are predicting,
it is leakage and must be dropped before any modeling.